# 10 - Artist collaboration network
**Goal:** build a graph where artists are nodes and shared tracks are edges; compute degree, betweenness and
Louvain communities; draw a static figure and export an interactive subgraph for the website.

**Inputs:** `data/processed/tracks.parquet` (artists list per track, canonical releases).

**Outputs:** figure `fig_10_01_*`, `reports/tables/tab_10_*.csv`, `data/processed/network.json`, `results.json -> network`.

In [1]:
import ast
import itertools
import json
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sonicatlas import DATA, TAB, SEED, results, theme

theme.apply()
a = pd.read_parquet(DATA / "processed" / "tracks.parquet",
                    columns=["artists", "n_artists", "is_canonical", "energy", "danceability", "acousticness", "valence", "year"])
multi = a[a.is_canonical & (a.n_artists >= 2)].copy()
multi["artists_list"] = multi["artists"].map(ast.literal_eval)
print(len(a), "tracks;", len(multi), "canonical collaborations")

1204025 tracks; 251382 canonical collaborations


In [2]:
edges = Counter()
for arts in multi["artists_list"]:
    for u, v in itertools.combinations(sorted(set(arts)), 2):
        edges[(u, v)] += 1
G = nx.Graph()
G.add_weighted_edges_from((u, v, w) for (u, v), w in edges.items())
comps = list(nx.connected_components(G))
giant = G.subgraph(max(comps, key=len)).copy()
print(G.number_of_nodes(), G.number_of_edges(), len(comps), giant.number_of_nodes())

84132 235196 6263 64214


## Centrality (betweenness approximated with 1,000 sampled sources) and Louvain communities

In [3]:
deg = dict(G.degree())
wdeg = dict(G.degree(weight="weight"))
btw = nx.betweenness_centrality(giant, k=min(1000, giant.number_of_nodes()), seed=SEED)
comm = nx.community.louvain_communities(giant, weight="weight", seed=SEED, resolution=1.0)
modularity = nx.community.modularity(giant, comm, weight="weight")
cmap = {n: i for i, c in enumerate(sorted(comm, key=len, reverse=True)) for n in c}
top = pd.DataFrame({"artist": list(deg), "degree": [deg[n] for n in deg], "collab_tracks": [wdeg[n] for n in deg],
                    "betweenness": [btw.get(n, 0.0) for n in deg], "community": [cmap.get(n, -1) for n in deg]})
top.sort_values("degree", ascending=False).head(20).to_csv(TAB / "tab_10_01_top_artists_degree.csv", index=False)
top.sort_values("betweenness", ascending=False).head(20).to_csv(TAB / "tab_10_02_top_artists_betweenness.csv", index=False)
print(len(comm), round(modularity, 3))
top.sort_values("degree", ascending=False).head(10)

189 0.828


,artist,degree,collab_tracks,betweenness,community
35,Johann Sebastian Bach,1066,21963,0.042349,25
1298,Traditional,1066,4497,0.111601,2
599,Wolfgang Amadeus Mozart,802,10133,0.046276,22
58,Ludwig van Beethoven,696,5695,0.021338,1
223,Anonymous,664,3104,0.033580,2
432,George Frideric Handel,601,9941,0.016101,7
636,Antonio Vivaldi,551,6194,0.021290,13
1310,Felix Mendelssohn,412,2486,0.005958,20
592,Slovak Radio Symphony Orchestra,409,6043,0.016807,15
575,Pyotr Ilyich Tchaikovsky,386,3635,0.008581,1


In [4]:
top.sort_values("betweenness", ascending=False).head(10)

,artist,degree,collab_tracks,betweenness,community
3634,Various Artists,344,946,0.151837,37
1298,Traditional,1066,4497,0.111601,2
599,Wolfgang Amadeus Mozart,802,10133,0.046276,22
35,Johann Sebastian Bach,1066,21963,0.042349,25
223,Anonymous,664,3104,0.033580,2
2797,Snoop Dogg,193,273,0.032520,0
44960,Alan Walker,58,145,0.027479,0
2139,Yukmouth,219,575,0.026483,0
327,Leonard Bernstein,301,2794,0.022390,12
7718,Paul Simon,42,54,0.021858,2


## Subgraph of the 300 most connected artists (static figure and website export)

In [5]:
keep = top.sort_values("degree", ascending=False).head(300)["artist"].tolist()
H = G.subgraph(keep).copy()
H = H.subgraph(max(nx.connected_components(H), key=len)).copy()  # largest connected piece only
pos = nx.spring_layout(H, seed=SEED, k=2.5 / np.sqrt(H.number_of_nodes()), iterations=200, weight=None)
fig, ax = plt.subplots(figsize=(11, 9))
comms_h = sorted({cmap.get(n, -1) for n in H}, key=lambda c: -sum(1 for n in H if cmap.get(n, -1) == c))
colour = {c: theme.OKABE_ITO[i % 8] for i, c in enumerate(comms_h)}
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.15, width=0.5)
for i, c in enumerate(comms_h[:8]):
    nodes = [n for n in H if cmap.get(n, -1) == c]
    nx.draw_networkx_nodes(H, pos, nodelist=nodes, node_size=[12 + 2.2 * np.sqrt(deg[n]) for n in nodes], node_color=colour[c],
                           node_shape="os^Dvph8"[i], ax=ax, alpha=0.85, label=f"community {c}")
other = [n for n in H if cmap.get(n, -1) not in comms_h[:8]]
nx.draw_networkx_nodes(H, pos, nodelist=other, node_size=10, node_color="#bbbbbb", ax=ax)
lab = {n: n for n in top[top.artist.isin(list(H))].sort_values("degree", ascending=False).head(10)["artist"]}
nx.draw_networkx_labels(H, pos, labels=lab, font_size=7, ax=ax, bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
ax.set_title(f"Most connected artists ({H.number_of_nodes()} in the largest component): {len(comm)} Louvain communities (modularity {modularity:.2f})")
ax.axis("off"); ax.legend(fontsize=8, markerscale=1.0, loc="upper left", bbox_to_anchor=(1.0, 1.0))
theme.save(fig, "fig_10_01_collaboration_network")

WindowsPath('D:/VIT/SEM 3/EDA/project/Team_05_25BDS0318_25BDS0308_SonicAtlas/reports/figures/fig_10_01_collaboration_network.png')

In [6]:
feat = multi.explode("artists_list").groupby("artists_list")[["energy", "danceability", "acousticness", "valence"]].mean()
nodes = [{"id": n, "degree": int(deg[n]), "tracks": int(wdeg[n]), "betweenness": round(float(btw.get(n, 0)), 6), "community": int(cmap.get(n, -1)),
          "x": round(float(pos[n][0]), 4), "y": round(float(pos[n][1]), 4),
          **{k: round(float(feat.loc[n, k]), 3) for k in feat.columns if n in feat.index}} for n in H]
links = [{"source": u, "target": v, "weight": int(d["weight"])} for u, v, d in H.edges(data=True)]
(DATA / "processed" / "network.json").write_text(json.dumps({"nodes": nodes, "links": links}))

t10d = top.sort_values("degree", ascending=False).head(5)
t10b = top.sort_values("betweenness", ascending=False).head(5)
results.save("network", {
    "collab_tracks": len(multi), "nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "components": len(comps),
    "giant_nodes": giant.number_of_nodes(), "giant_share_pct": 100 * giant.number_of_nodes() / G.number_of_nodes(),
    "communities": len(comm), "modularity": modularity, "density": nx.density(G),
    "top_degree": t10d[["artist", "degree"]].values.tolist(), "top_betweenness": t10b[["artist", "betweenness"]].values.tolist(),
    "web_nodes": len(nodes), "web_links": len(links), "betweenness_samples": min(1000, giant.number_of_nodes()),
})